In [ ]:
!pip install mediapipe==0.10.14

In [1]:
# define gestures to detect

num_classes = 5
class_labels = ['feast', 'palm', 'thumbs-down', 'thumbs-up', 'victory']


In [2]:
import cv2

In [3]:
# import mediapipe library

import mediapipe as mp
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

In [5]:
# hand model
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1)

In [ ]:
!unzip gestures.zip

In [10]:
import os
import numpy as np

x = []
y = []

# build dataset by loading image files for each gesture
for gesture in class_labels:
  dir_path = os.path.join("gestures", gesture)
  for item in os.listdir(dir_path):
    image_file = os.path.join(dir_path, item)
    if os.path.isfile(image_file):
      image = cv2.imread(image_file)

      if image is not None:

        # convert image to RGB
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # get landmarks
        output = hands.process(rgb_image)
        if output.multi_hand_landmarks:
          y.append(gesture)

          # extract co-ordinates of hand landmark points
          landmarks = []
          for landmark in output.multi_hand_landmarks[0].landmark:
            landmarks.append(landmark.x)
            landmarks.append(landmark.y)
            landmarks.append(landmark.z)

          for landmark in output.multi_hand_world_landmarks[0].landmark:
            landmarks.append(landmark.x)
            landmarks.append(landmark.y)
            landmarks.append(landmark.z)

          x.append(landmarks)

# x stores landmark points
x = np.array(x)

# y stores corresponding class name
y = np.array(y)

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [11]:
x.shape

(1312, 126)

In [12]:
y.shape

(1312,)

In [13]:
# split data for training and testing

from sklearn.model_selection import train_test_split
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size=0.2)

In [14]:
# choose and run ML model

from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier().fit(train_x, train_y)
model.score(test_x, test_y)

1.0

In [15]:
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier().fit(train_x, train_y)
model.score(test_x, test_y)

1.0

In [17]:
# define neural network architecture

import keras
from keras import layers

inputs = keras.Input(shape=(126, ))
features = layers.Dense(64, activation="linear")(inputs)
features = layers.Dense(64, activation="linear")(features)
features = layers.Dense(64, activation="linear")(features)

outputs = layers.Dense(num_classes, activation="softmax")(features)
model = keras.Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 126)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,773 (65.52 KB)

 Trainable params: 16,773 (65.52 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

In [19]:
# to convert class labels from string to number

from sklearn.preprocessing import LabelEncoder
z = LabelEncoder().fit_transform(y)

In [20]:
model.fit(x, z, validation_split=0.2, epochs=10, shuffle=True)

Epoch 1/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.6435 - loss: 1.0419 - val_accuracy: 0.1597 - val_loss: 5.9577
Epoch 2/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0946 - val_accuracy: 0.1597 - val_loss: 7.9201
Epoch 3/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0109 - val_accuracy: 0.1597 - val_loss: 8.3874
Epoch 4/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0048 - val_accuracy: 0.1597 - val_loss: 8.5966
Epoch 5/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0030 - val_accuracy: 0.1597 - val_loss: 8.8240
Epoch 6/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0021 - val_accuracy: 0.1597 - val_loss: 8.9229
Epoch 7/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0015 - val_accuracy: 0.1597 - val_loss: 9.0506
Epoch 8/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0012 - val_accuracy: 0.1597 - val_loss